# Modul 15: TensorFlow-Grundlagen und dichte Keras-Modelle

    **Notebooktyp:** Übungen mit ausführlichen Lösungen  
    **Vorlesungen dieses Moduls:** TensorFlow Grundlagen, Dichte Keras-Modelle  
    **Erwarteter Schwierigkeitsgrad:** Mittlere bis fortgeschrittene Framework-Anwendung  
    **Orientierungszeit:** etwa 130 bis 180 Minuten

    ## Überblick

    Sie arbeiten mit TensorFlow-Tensoren, automatischen Gradienten und tf.data. Danach bauen, trainieren, bewerten und speichern Sie ein kleines dichtes Keras-Modell für eine binäre Klassifikationsaufgabe.

    ## Verwendete Vorlesungsnotebooks

    Die Aufgaben wurden aus dem Inhalt beider Vorlesungen dieses Moduls abgeleitet:

    - `ML Für Anfänger - Record_Module_15A_20260723.ipynb`
- `ML Für Anfänger - Record_Module_15B_20260723.ipynb`

    ## Colab-Kompatibilität

    Dieses Notebook ist für die kostenlose Version von Google Colab ausgelegt. Die Daten sind eingebaut, synthetisch erzeugt oder öffentlich verfügbar. Modelle und Trainingsbudgets sind bewusst klein gehalten. Führen Sie die Zellen in der vorgegebenen Reihenfolge aus.

## Lernziele

    Nach der Bearbeitung sollen Sie:

    - Tensorformen, Datentypen, NumPy-Konvertierung und Broadcasting sicher handhaben.
- Gradienten mit GradientTape berechnen und mit einer manuellen Ableitung vergleichen.
- tf.data-Datasets mit Mapping, Shuffling, Batching und Prefetching vorbereiten.
- Sequential-Modelle mit zur Aufgabe passenden Eingabe- und Ausgabeschichten erstellen.
- Loss, Optimierer und Metriken passend zur binären Klassifikation wählen.
- Trainingsverläufe, Vorhersagewahrscheinlichkeiten und Generalisierung interpretieren.
- Regularisierung, Baseline-Vergleich sowie Speichern und Laden in einem Mini-Projekt verbinden.

    ## Bewertete Fähigkeiten

    - TensorFlow-Tensoren, Broadcasting und GradientTape
- tf.data mit map, shuffle, batch und prefetch
- Keras Sequential, compile, fit, evaluate und predict
- Early Stopping, L2-Regularisierung und Dropout
- Baseline-Vergleich und reproduzierbares Speichern im Keras-Format

## Arbeitsanweisungen

Dieses Lösungsnotebook entspricht dem Übungsnotebook Aufgabe für Aufgabe. Führen Sie es von oben nach unten aus und vergleichen Sie nicht nur Endwerte, sondern auch Vorgehen, Formprüfungen, Datenaufteilung und Interpretation. Die Kommentare erklären bewusst auch typische Fehlerquellen und methodische Entscheidungen.

- Führen Sie zuerst das gemeinsame Setup aus.
- Verändern Sie vorgegebene Splits und Seeds nur, wenn eine Aufgabe dies ausdrücklich erlaubt.
- Prüfen Sie Formen, Datentypen und Wertebereiche frühzeitig.
- Begründen Sie Modell-, Metrik- und Visualisierungsentscheidungen.
- Achten Sie auf Datenleckage und eine saubere Trennung von Training, Validierung und Test.

## Gemeinsames Setup

Führen Sie diese Zelle einmal aus, bevor Sie mit Aufgabe 1 beginnen.

In [ ]:
# TensorFlow ist in Google Colab üblicherweise bereits verfügbar.
# Der Fallback installiert nur dann eine CPU-Version, wenn der Import fehlt.
import os
import sys
import subprocess
import warnings
from pathlib import Path

try:
    import tensorflow as tf
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tensorflow-cpu"])
    import tensorflow as tf

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42
FAST_MODE = os.environ.get("COURSE_FAST", "0") == "1"
OFFLINE_MODE = os.environ.get("COURSE_OFFLINE", "0") == "1"

np.random.seed(RANDOM_SEED)
tf.keras.utils.set_random_seed(RANDOM_SEED)
warnings.filterwarnings("ignore", category=FutureWarning)

import tempfile

from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Ein kleiner tabellarischer Datensatz wird vollständig lokal erzeugt.
X_15, y_15 = make_classification(
    n_samples=900,
    n_features=12,
    n_informative=7,
    n_redundant=2,
    weights=[0.62, 0.38],
    class_sep=1.0,
    flip_y=0.06,
    random_state=RANDOM_SEED,
)
X_train_valid_15, X_test_15, y_train_valid_15, y_test_15 = train_test_split(
    X_15,
    y_15,
    test_size=0.20,
    stratify=y_15,
    random_state=RANDOM_SEED,
)
X_train_15, X_valid_15, y_train_15, y_valid_15 = train_test_split(
    X_train_valid_15,
    y_train_valid_15,
    test_size=0.25,
    stratify=y_train_valid_15,
    random_state=RANDOM_SEED,
)

# Die Standardisierung lernt ausschließlich aus dem Training.
scaler_15 = StandardScaler()
X_train_15 = scaler_15.fit_transform(X_train_15).astype("float32")
X_valid_15 = scaler_15.transform(X_valid_15).astype("float32")
X_test_15 = scaler_15.transform(X_test_15).astype("float32")
y_train_15 = y_train_15.astype("float32")
y_valid_15 = y_valid_15.astype("float32")
y_test_15 = y_test_15.astype("float32")

print("Train/Valid/Test:", X_train_15.shape, X_valid_15.shape, X_test_15.shape)

print("TensorFlow-Version:", tf.__version__)
print("Schneller Validierungsmodus:", FAST_MODE)


## Aufgabe 1: Tensoren, Broadcasting und automatische Gradienten

    Untersuchen Sie TensorFlow-Tensoren und kontrollieren Sie eine automatische Ableitung.

1. Erzeugen Sie einen Skalar, einen Vektor, eine Matrix und einen kleinen Bildstapel als `tf.Tensor`.
2. Geben Sie jeweils Form, Rang und Datentyp aus und konvertieren Sie die Matrix zurück nach NumPy.
3. Standardisieren Sie jede Spalte einer `3 x 2`-Matrix durch Broadcasting mit vorgegebenen Mittelwerten und Standardabweichungen.
4. Verwenden Sie `tf.GradientTape`, um für `loss(w) = mean((w*x - y)^2)` den Gradienten nach `w` zu berechnen.
5. Leiten Sie denselben Gradienten manuell ab und vergleichen Sie beide Werte numerisch.

> **Hinweis:** Prüfen Sie bei Broadcasting zuerst die Form der letzten Achse.

In [ ]:
feature_matrix = tf.constant(
    [[2.0, 10.0], [4.0, 14.0], [6.0, 18.0]],
    dtype=tf.float32,
)
feature_means = tf.constant([4.0, 14.0], dtype=tf.float32)
feature_stds = tf.constant([2.0, 4.0], dtype=tf.float32)

x_gradient = tf.constant([1.0, 2.0, 3.0], dtype=tf.float32)
y_gradient = tf.constant([2.0, 4.0, 5.0], dtype=tf.float32)

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Tensoren, Broadcasting und automatische Gradienten
#
# Ziel dieser Codezelle:
# Untersuchen Sie TensorFlow-Tensoren und kontrollieren Sie eine automatische
# Ableitung. 1. Erzeugen Sie einen Skalar, einen Vektor, eine Matrix und einen
# kleinen Bildstapel als tf.Tensor. 2. Geben Sie jeweils Form, Ran...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

feature_matrix = tf.constant(
    [[2.0, 10.0], [4.0, 14.0], [6.0, 18.0]],
    dtype=tf.float32,
)
feature_means = tf.constant([4.0, 14.0], dtype=tf.float32)
feature_stds = tf.constant([2.0, 4.0], dtype=tf.float32)

x_gradient = tf.constant([1.0, 2.0, 3.0], dtype=tf.float32)
y_gradient = tf.constant([2.0, 4.0, 5.0], dtype=tf.float32)

# Tensoren können null, eine oder viele Achsen besitzen.
scalar = tf.constant(3.5, dtype=tf.float32)
vector = tf.constant([1, 2, 3], dtype=tf.int32)
matrix = tf.constant([[1.0, 2.0], [3.0, 4.0]], dtype=tf.float32)
image_batch = tf.zeros((4, 8, 8, 1), dtype=tf.float32)

named_tensors = {
    "Skalar": scalar,
    "Vektor": vector,
    "Matrix": matrix,
    "Bildstapel": image_batch,
}
for name, tensor in named_tensors.items():
    # tf.rank liefert einen Tensor. Für die kompakte Ausgabe wird
    # dessen Wert mit numpy() in eine Python-kompatible Zahl gebracht.
    print(
        name,
        "Form:", tensor.shape,
        "Rang:", int(tf.rank(tensor).numpy()),
        "Datentyp:", tensor.dtype.name,
    )

matrix_as_numpy = matrix.numpy()
print("Matrix als NumPy-Array:\n", matrix_as_numpy)

# Die Vektoren der Form (2,) werden automatisch auf jede Tabellenzeile
# angewendet. Das ist spaltenweises Broadcasting.
standardized = (feature_matrix - feature_means) / feature_stds
expected = tf.constant([[-1.0, -1.0], [0.0, 0.0], [1.0, 1.0]])
tf.debugging.assert_near(standardized, expected)
print("Standardisiert:\n", standardized.numpy())

# Variable markiert w als trainierbaren Wert. GradientTape zeichnet
# die Tensoroperationen innerhalb des Kontextes auf.
w = tf.Variable(1.5, dtype=tf.float32)
with tf.GradientTape() as tape:
    predictions = w * x_gradient
    loss = tf.reduce_mean(tf.square(predictions - y_gradient))
automatic_gradient = tape.gradient(loss, w)

# Für mean((w*x-y)^2) lautet die Ableitung
# mean(2 * (w*x-y) * x).
manual_gradient = tf.reduce_mean(
    2.0 * (w * x_gradient - y_gradient) * x_gradient
)
tf.debugging.assert_near(automatic_gradient, manual_gradient)

print("Verlust:", round(float(loss.numpy()), 5))
print("GradientTape-Gradient:", round(float(automatic_gradient.numpy()), 5))
print("Manueller Gradient:", round(float(manual_gradient.numpy()), 5))

### Reflexion zu Aufgabe 1

Tensorformen bestimmen, welche Operationen zulässig sind. Broadcasting ist hilfreich, kann aber bei unpassenden Achsen unbemerkt eine andere Rechnung erzeugen. `GradientTape` verfolgt differenzierbare Operationen und berechnet mit der Kettenregel den Gradienten nach beobachteten Variablen. Der manuelle Vergleich ist besonders bei kleinen Testfällen ein gutes Diagnosewerkzeug.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 2: Eine tf.data-Pipeline aufbauen

    Erstellen Sie eine reproduzierbare Eingabepipeline aus den Trainingsdaten.

1. Erzeugen Sie ein Dataset aus Merkmalen und Labels.
2. Schreiben Sie eine Mapping-Funktion, die die Merkmale leicht mit deterministischem Faktor skaliert und das Label in die Form `(1,)` bringt.
3. Mischen Sie nur die Trainingsdaten mit einem festen Seed.
4. Bilden Sie Batches der Größe 32 und verwenden Sie `prefetch(tf.data.AUTOTUNE)`.
5. Erstellen Sie getrennte Validierungs- und Test-Datasets ohne Shuffling.
6. Untersuchen Sie einen Batch und bestätigen Sie Formen und Datentypen.

> **Hinweis:** Erzeugen Sie erst einen funktionierenden Dataset-Batch und ergänzen Sie danach Prefetching.

In [ ]:
batch_size_15 = 32

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Eine tf.data-Pipeline aufbauen
#
# Ziel dieser Codezelle:
# Erstellen Sie eine reproduzierbare Eingabepipeline aus den Trainingsdaten. 1.
# Erzeugen Sie ein Dataset aus Merkmalen und Labels. 2. Schreiben Sie eine Mapping-
# Funktion, die die Merkmale leicht mit deterministischem Fa...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

batch_size_15 = 32

def prepare_example(features, label):
    # Eine TensorFlow-Mapping-Funktion sollte nur Tensoroperationen
    # verwenden. Die kleine Skalierung dient hier als sichtbare,
    # deterministische Transformation.
    prepared_features = tf.cast(features, tf.float32) * tf.constant(1.0, tf.float32)
    prepared_label = tf.reshape(tf.cast(label, tf.float32), (1,))
    return prepared_features, prepared_label

# from_tensor_slices trennt die erste Achse in einzelne Beispiele.
train_dataset_15 = tf.data.Dataset.from_tensor_slices((X_train_15, y_train_15))
train_dataset_15 = train_dataset_15.map(
    prepare_example,
    num_parallel_calls=tf.data.AUTOTUNE,
)
train_dataset_15 = train_dataset_15.shuffle(
    buffer_size=len(X_train_15),
    seed=RANDOM_SEED,
    reshuffle_each_iteration=True,
)
train_dataset_15 = train_dataset_15.batch(batch_size_15).prefetch(tf.data.AUTOTUNE)

# Validierung und Test behalten ihre Reihenfolge. Mischen ist für die
# Bewertung unnötig und erschwert die Zuordnung einzelner Fehler.
valid_dataset_15 = tf.data.Dataset.from_tensor_slices((X_valid_15, y_valid_15))
valid_dataset_15 = valid_dataset_15.map(prepare_example).batch(batch_size_15).prefetch(tf.data.AUTOTUNE)

test_dataset_15 = tf.data.Dataset.from_tensor_slices((X_test_15, y_test_15))
test_dataset_15 = test_dataset_15.map(prepare_example).batch(batch_size_15).prefetch(tf.data.AUTOTUNE)

first_features, first_labels = next(iter(train_dataset_15))
assert first_features.shape[1] == X_train_15.shape[1]
assert first_labels.shape[1] == 1
assert first_features.dtype == tf.float32
assert first_labels.dtype == tf.float32

print("Merkmalsbatch:", first_features.shape, first_features.dtype)
print("Labelbatch:", first_labels.shape, first_labels.dtype)
print("Erste drei Labels:", first_labels[:3].numpy().ravel().tolist())

### Reflexion zu Aufgabe 2

Eine `tf.data`-Pipeline trennt Datenspeicherung, Transformation und Batching. Shuffling gehört in der Regel nur zum Training. Mapping muss die Zuordnung zwischen Merkmalen und Labels erhalten. Prefetching kann Datenvorbereitung und Modellrechnung überlappen, ohne dass eine große Datenmenge gleichzeitig im Speicher liegen muss.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 3: Ein passendes Sequential-Modell definieren und kompilieren

    Erstellen Sie ein dichtes Keras-Modell für die vorbereitete binäre Klassifikation.

1. Verwenden Sie eine ausdrückliche Eingabeschicht für zwölf Merkmale.
2. Bauen Sie zwei kleine Dense-Schichten mit ReLU-Aktivierung.
3. Verwenden Sie genau ein Ausgabeneuron mit Sigmoid.
4. Kompilieren Sie das Modell mit Adam, binärer Kreuzentropie, Accuracy und AUC.
5. Geben Sie eine Modellzusammenfassung aus und prüfen Sie die Ausgabeform für einen Mini-Batch.
6. Begründen Sie, warum eine lineare Ausgabe mit MSE oder eine zehnklassige Softmax hier unpassend wäre.

> **Hinweis:** Leiten Sie die Zahl der Ausgabeneuronen direkt aus der Zielcodierung ab.

In [ ]:
# Speichern Sie das Modell unter dem Namen model_15, damit die
# folgenden Aufgaben es weiterverwenden können.

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Ein passendes Sequential-Modell definieren und kompilieren
#
# Ziel dieser Codezelle:
# Erstellen Sie ein dichtes Keras-Modell für die vorbereitete binäre Klassifikation.
# 1. Verwenden Sie eine ausdrückliche Eingabeschicht für zwölf Merkmale. 2. Bauen
# Sie zwei kleine Dense-Schichten mit ReLU-Aktivierung....
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

# Der feste Seed vor der Modellerstellung reproduziert die
# Initialisierung der trainierbaren Gewichte.
tf.keras.utils.set_random_seed(RANDOM_SEED)

model_15 = tf.keras.Sequential(
    [
        # Input beschreibt die Form eines einzelnen Beispiels. Die
        # Batchdimension wird von Keras automatisch ergänzt.
        tf.keras.layers.Input(shape=(X_train_15.shape[1],), name="features"),
        tf.keras.layers.Dense(24, activation="relu", name="hidden_1"),
        tf.keras.layers.Dense(12, activation="relu", name="hidden_2"),
        # Ein Sigmoid-Neuron liefert P(Klasse 1) für jedes Beispiel.
        tf.keras.layers.Dense(1, activation="sigmoid", name="probability"),
    ],
    name="dense_binary_classifier",
)

model_15.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="auc"),
    ],
)

model_15.summary()
example_output = model_15(X_train_15[:5], training=False)
assert example_output.shape == (5, 1)
assert bool(tf.reduce_all((example_output >= 0.0) & (example_output <= 1.0)))
print("Ausgabeform:", example_output.shape)
print("Untrainierte Wahrscheinlichkeiten:", np.round(example_output.numpy().ravel(), 3))

### Reflexion zu Aufgabe 3

Bei binärer Klassifikation mit Labels 0 und 1 passt ein Sigmoid-Neuron zur binären Kreuzentropie. Eine lineare Ausgabe wäre nicht auf den Wahrscheinlichkeitsbereich begrenzt, und MSE ist für diese probabilistische Klassifikationsaufgabe weniger passend. Zehn Softmax-Neuronen würden eine zehnklassige Zielcodierung voraussetzen, die hier nicht existiert. Architektur, Labeldarstellung, Aktivierung und Loss müssen gemeinsam geplant werden.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 4: Trainieren, Lernkurven prüfen und Vorhersagen interpretieren

    Trainieren und bewerten Sie `model_15`.

1. Verwenden Sie das Trainings- und Validierungs-Dataset aus Aufgabe 2.
2. Trainieren Sie mit `EarlyStopping`, überwachen Sie `val_loss` und stellen Sie die besten Gewichte wieder her.
3. Visualisieren Sie Loss, Accuracy und AUC für Training und Validierung.
4. Bewerten Sie das Modell genau einmal auf dem Test-Dataset.
5. Wandeln Sie Testwahrscheinlichkeiten mit Schwelle 0.5 in Klassen um und erstellen Sie eine Konfusionsmatrix.
6. Zeigen Sie fünf Beispiele mit ihrer Wahrscheinlichkeit, Vorhersage und wahrem Label.

> **Hinweis:** Nutzen Sie die Schlüssel von `history.history`, statt Metriknamen zu erraten.

In [ ]:
epochs_15 = 8 if FAST_MODE else 50

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Trainieren, Lernkurven prüfen und Vorhersagen interpretieren
#
# Ziel dieser Codezelle:
# Trainieren und bewerten Sie model15. 1. Verwenden Sie das Trainings- und
# Validierungs-Dataset aus Aufgabe 2. 2. Trainieren Sie mit EarlyStopping,
# überwachen Sie valloss und stellen Sie die besten Gewichte wieder her....
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

epochs_15 = 8 if FAST_MODE else 50

early_stopping_15 = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=6,
    min_delta=1e-4,
    restore_best_weights=True,
)

# verbose=0 hält die Notebookausgabe kompakt. Alle Werte bleiben im
# History-Objekt für die spätere Analyse verfügbar.
history_15 = model_15.fit(
    train_dataset_15,
    validation_data=valid_dataset_15,
    epochs=epochs_15,
    callbacks=[early_stopping_15],
    verbose=0,
)
history_frame_15 = pd.DataFrame(history_15.history)
print("Trainierte Epochen:", len(history_frame_15))
print(history_frame_15.tail(3).round(4))

# Jede Metrik erhält eine eigene Abbildung, damit ihre Skala klar ist.
for metric_name, label in [
    ("loss", "Loss"),
    ("accuracy", "Accuracy"),
    ("auc", "AUC"),
]:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(history_frame_15[metric_name], label="Training")
    ax.plot(history_frame_15[f"val_{metric_name}"], label="Validierung")
    ax.set_title(f"{label} während des Trainings")
    ax.set_xlabel("Epoche")
    ax.set_ylabel(label)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Das Testset wird erst nach Modellauswahl und Early Stopping geprüft.
test_results_15 = model_15.evaluate(test_dataset_15, verbose=0, return_dict=True)
test_probabilities_15 = model_15.predict(test_dataset_15, verbose=0).ravel()
test_predictions_15 = (test_probabilities_15 >= 0.5).astype(int)
confusion_15 = confusion_matrix(y_test_15.astype(int), test_predictions_15)
balanced_15 = balanced_accuracy_score(y_test_15.astype(int), test_predictions_15)

print("Testergebnisse:", {key: round(float(value), 4) for key, value in test_results_15.items()})
print("Test Balanced Accuracy:", round(float(balanced_15), 4))
print("Konfusionsmatrix:\n", confusion_15)

preview_15 = pd.DataFrame(
    {
        "probability_class_1": test_probabilities_15[:5],
        "prediction": test_predictions_15[:5],
        "true_label": y_test_15[:5].astype(int),
    }
)
print(preview_15.round(3).to_string(index=False))

### Reflexion zu Aufgabe 4

Trainingsmetriken allein reichen nicht zur Bewertung. Eine wachsende Lücke zwischen Training und Validierung kann auf Overfitting hinweisen. Accuracy und AUC beantworten unterschiedliche Fragen, und die Konfusionsmatrix zeigt, welche Fehlerarten entstehen. Die Schwelle 0.5 ist ein Ausgangspunkt, aber keine universell optimale Entscheidung. Sie sollte anhand fachlicher Fehlerkosten und Validierungsdaten gewählt werden, nicht anhand des Testsets.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 5: Integration: Regularisierung, Baseline und Modellartefakt

    Erstellen Sie ein zweites, reguliertes Modell und schließen Sie den Workflow reproduzierbar ab.

1. Trainieren Sie eine `DummyClassifier`-Baseline nur auf den Trainingsdaten und bewerten Sie sie auf der Validierung.
2. Erstellen Sie ein kleineres Keras-Modell mit L2-Regularisierung und Dropout.
3. Trainieren Sie es mit denselben Trainings- und Validierungsdaten sowie Early Stopping.
4. Vergleichen Sie Baseline, ursprüngliches Modell und reguliertes Modell mit Balanced Accuracy auf denselben Validierungsdaten.
5. Wählen Sie anhand der Validierung das bessere Keras-Modell und bewerten Sie dieses auf dem Testset.
6. Speichern Sie das ausgewählte Modell im `.keras`-Format in einem temporären Ordner, laden Sie es neu und bestätigen Sie identische Referenzvorhersagen.

> **Hinweis:** Wählen Sie das Modell auf der Validierung und berichten Sie den Testwert erst danach.

In [ ]:
regularized_epochs_15 = 8 if FAST_MODE else 50

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Integration: Regularisierung, Baseline und Modellartefakt
#
# Ziel dieser Codezelle:
# Erstellen Sie ein zweites, reguliertes Modell und schließen Sie den Workflow
# reproduzierbar ab. 1. Trainieren Sie eine DummyClassifier-Baseline nur auf den
# Trainingsdaten und bewerten Sie sie auf der Validierung. 2. E...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

regularized_epochs_15 = 8 if FAST_MODE else 50

# Die Baseline kennt keine Merkmalsmuster und sagt die häufigste
# Trainingsklasse voraus. Sie setzt eine sinnvolle Mindestleistung.
baseline_15 = DummyClassifier(strategy="most_frequent")
baseline_15.fit(X_train_15, y_train_15.astype(int))
baseline_valid_predictions_15 = baseline_15.predict(X_valid_15)
baseline_valid_balanced_15 = balanced_accuracy_score(
    y_valid_15.astype(int),
    baseline_valid_predictions_15,
)

tf.keras.utils.set_random_seed(RANDOM_SEED + 1)
regularized_model_15 = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(X_train_15.shape[1],)),
        # L2 begrenzt große Kernelgewichte. Dropout deaktiviert nur
        # während des Trainings zufällig einen Teil der Aktivierungen.
        tf.keras.layers.Dense(
            16,
            activation="relu",
            kernel_regularizer=tf.keras.regularizers.l2(0.002),
        ),
        tf.keras.layers.Dropout(0.20),
        tf.keras.layers.Dense(
            8,
            activation="relu",
            kernel_regularizer=tf.keras.regularizers.l2(0.002),
        ),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ],
    name="regularized_binary_classifier",
)
regularized_model_15.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

regularized_model_15.fit(
    train_dataset_15,
    validation_data=valid_dataset_15,
    epochs=regularized_epochs_15,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=6,
            restore_best_weights=True,
        )
    ],
    verbose=0,
)

# Für einen fairen Vergleich werden beide Netze auf exakt denselben
# Validierungsbeispielen und mit derselben Schwelle bewertet.
original_valid_probabilities_15 = model_15.predict(X_valid_15, verbose=0).ravel()
regularized_valid_probabilities_15 = regularized_model_15.predict(X_valid_15, verbose=0).ravel()
original_valid_balanced_15 = balanced_accuracy_score(
    y_valid_15.astype(int),
    (original_valid_probabilities_15 >= 0.5).astype(int),
)
regularized_valid_balanced_15 = balanced_accuracy_score(
    y_valid_15.astype(int),
    (regularized_valid_probabilities_15 >= 0.5).astype(int),
)

comparison_15 = pd.DataFrame(
    {
        "model": ["Dummy-Baseline", "ursprüngliches Keras-Modell", "reguliertes Keras-Modell"],
        "validation_balanced_accuracy": [
            baseline_valid_balanced_15,
            original_valid_balanced_15,
            regularized_valid_balanced_15,
        ],
    }
).sort_values("validation_balanced_accuracy", ascending=False)
print(comparison_15.round(4).to_string(index=False))

# Nur die beiden trainierbaren Keras-Modelle kommen für das Artefakt
# infrage. Die Wahl nutzt ausschließlich die Validierungsleistung.
if regularized_valid_balanced_15 >= original_valid_balanced_15:
    selected_model_15 = regularized_model_15
    selected_name_15 = "reguliertes Keras-Modell"
else:
    selected_model_15 = model_15
    selected_name_15 = "ursprüngliches Keras-Modell"

selected_test_probabilities_15 = selected_model_15.predict(X_test_15, verbose=0).ravel()
selected_test_predictions_15 = (selected_test_probabilities_15 >= 0.5).astype(int)
selected_test_balanced_15 = balanced_accuracy_score(
    y_test_15.astype(int),
    selected_test_predictions_15,
)
print("Ausgewählt:", selected_name_15)
print("Test Balanced Accuracy:", round(float(selected_test_balanced_15), 4))

# Ein temporärer Ordner verhindert hardcodierte lokale Pfade und wird
# nach der Prüfung automatisch entfernt.
reference_features_15 = X_test_15[:12]
reference_predictions_15 = selected_model_15.predict(reference_features_15, verbose=0)
with tempfile.TemporaryDirectory() as temporary_directory:
    model_path_15 = Path(temporary_directory) / "module_15_classifier.keras"
    selected_model_15.save(model_path_15)
    loaded_model_15 = tf.keras.models.load_model(model_path_15)
    loaded_predictions_15 = loaded_model_15.predict(reference_features_15, verbose=0)

np.testing.assert_allclose(
    reference_predictions_15,
    loaded_predictions_15,
    rtol=1e-6,
    atol=1e-7,
)
print("Speichern/Laden geprüft: Referenzvorhersagen sind identisch.")

### Reflexion zu Aufgabe 5

Der Vergleich ist nur fair, weil alle Ansätze dieselben Datenpartitionen und dieselbe Validierungsmetrik verwenden. Regularisierung ist kein garantierter Leistungsgewinn, sondern eine kontrollierte Einschränkung der Modellkomplexität. Die finale Modellwahl gehört zur Validierungsphase. Erst danach liefert das Testset eine einmalige, möglichst unverzerrte Schätzung. Ein gespeichertes Modell sollte zusammen mit Vorverarbeitung, Schwelle, Versionsinformationen und Referenztests dokumentiert werden.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Abschluss und Selbstkontrolle

Prüfen Sie nach dem Durcharbeiten, ob Sie jede Lösung ohne bloßes Kopieren erklären könnten. Achten Sie besonders auf die Stellen, an denen Datenleckage, unpassende Formen, falsche Metriken oder unkontrollierte Zufälligkeit zu scheinbar guten, aber methodisch falschen Ergebnissen führen könnten.

- Alle Aufgaben und Unterpunkte wurden bearbeitet.
- Verwendete Seeds und Datenpartitionen sind nachvollziehbar.
- Testdaten wurden nicht vorzeitig für Entscheidungen genutzt.
- Ergebnisse werden vorsichtig und fachlich begründet interpretiert.
- Es gibt keine hardcodierten lokalen Dateipfade oder privaten Zugangsdaten.